# ROGII private sequence TCN well-balanced pilot

TCN v2 improved pooled RMSE but failed the frozen well-coverage gate. This single fold-0 qualification changes only training-well sampling from suffix-length-proportional to uniform by well. Architecture, features, seed, steps, 0.20 overlay, validation partition, and all decision gates remain frozen. It writes an OOF report and checkpoint only; it cannot create a submission.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

candidates = [
    Path('/kaggle/input/rogii-private-sequence-pilot-cache-v1'),
    Path('/kaggle/input/datasets/heralces/rogii-private-sequence-pilot-cache-v1'),
]
cache = next((p for p in candidates if (p / 'sequence_cache_metadata.json').exists()), None)
if cache is None:
    raise RuntimeError(f'private sequence cache not found: {candidates}')
source = '"""Run one preregistered GPU TCN residual pilot against the competitive OOF base.\n\nThis is deliberately an experiment, not a submission builder.  The primary\ncandidate is a fixed 0.20 overlay of a bounded TCN residual prediction.  A\nsingle strict spatial fold is used as the cheap first gate; full multi-fold\ntraining is authorized only if this pilot passes.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport math\nimport random\nimport time\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport torch\nfrom torch import nn\n\nPRIMARY_SCALE = 0.20\nDIAGNOSTIC_SCALES = (0.10, 0.20, 0.30)\nTARGET_CLIP = 50.0\nOUTPUT_BOUND = 30.0\nWINDOW_ROWS = 256\nBATCH_SIZE = 32\nTRAINING_STEPS = 4_000\nLEARNING_RATE = 8e-4\nWEIGHT_DECAY = 1e-4\nRANDOM_SEED = 20_260_727\nPROTOCOL = "spatial_cluster_7_seed_20260727"\nVALIDATION_FOLD = 0\n\n\ndef _parser() -> argparse.ArgumentParser:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--cache-dir", type=Path, required=True)\n    parser.add_argument("--output", type=Path, default=Path("sequence_tcn_pilot_report.json"))\n    parser.add_argument("--checkpoint", type=Path, default=Path("sequence_tcn_pilot.pt"))\n    parser.add_argument("--steps", type=int, default=TRAINING_STEPS)\n    parser.add_argument("--batch-size", type=int, default=BATCH_SIZE)\n    parser.add_argument("--window-rows", type=int, default=WINDOW_ROWS)\n    parser.add_argument(\n        "--well-sampling",\n        choices=("length", "uniform"),\n        default="length",\n        help=(\n            "Choose training wells proportional to suffix length (original pilot) "\n            "or uniformly by well."\n        ),\n    )\n    parser.add_argument("--device", choices=("auto", "cpu", "cuda"), default="auto")\n    parser.add_argument(\n        "--smoke-wells",\n        type=int,\n        default=0,\n        help="Limit both partitions for a local code-path smoke test; zero uses the full fold.",\n    )\n    return parser\n\n\ndef _json_safe(value: Any) -> Any:\n    if isinstance(value, dict):\n        return {str(key): _json_safe(item) for key, item in value.items()}\n    if isinstance(value, (list, tuple)):\n        return [_json_safe(item) for item in value]\n    if isinstance(value, np.ndarray):\n        return _json_safe(value.tolist())\n    if isinstance(value, np.generic):\n        return value.item()\n    if isinstance(value, float) and not math.isfinite(value):\n        return None\n    return value\n\n\ndef _rmse(truth: np.ndarray, prediction: np.ndarray) -> float:\n    return float(np.sqrt(np.mean(np.square(truth - prediction))))\n\n\nclass ResidualBlock(nn.Module):\n    def __init__(self, channels: int, dilation: int) -> None:\n        super().__init__()\n        self.network = nn.Sequential(\n            nn.Conv1d(\n                channels,\n                channels,\n                kernel_size=5,\n                padding=2 * dilation,\n                dilation=dilation,\n            ),\n            nn.GroupNorm(8, channels),\n            nn.GELU(),\n            nn.Dropout(0.08),\n            nn.Conv1d(channels, channels, kernel_size=1),\n            nn.GroupNorm(8, channels),\n        )\n        self.activation = nn.GELU()\n\n    def forward(self, values: torch.Tensor) -> torch.Tensor:\n        return self.activation(values + self.network(values))\n\n\nclass SequenceResidualTCN(nn.Module):\n    def __init__(self, input_features: int, channels: int = 64) -> None:\n        super().__init__()\n        self.input = nn.Conv1d(input_features, channels, kernel_size=1)\n        self.blocks = nn.Sequential(\n            *(ResidualBlock(channels, dilation) for dilation in (1, 2, 4, 8, 16, 32))\n        )\n        self.output = nn.Conv1d(channels, 1, kernel_size=1)\n\n    def forward(self, values: torch.Tensor) -> torch.Tensor:\n        raw = self.output(self.blocks(self.input(values))).squeeze(1)\n        return OUTPUT_BOUND * torch.tanh(raw / OUTPUT_BOUND)\n\n\ndef _load_cache(cache_dir: Path) -> dict[str, Any]:\n    metadata = json.loads(\n        (cache_dir / "sequence_cache_metadata.json").read_text(encoding="utf-8")\n    )\n    features = np.load(cache_dir / "features.float16.npy", mmap_mode="r", allow_pickle=False)\n    base = np.load(\n        cache_dir / "competitive_base.float32.npy",\n        mmap_mode="r",\n        allow_pickle=False,\n    )\n    truth = np.load(cache_dir / "truth.float32.npy", mmap_mode="r", allow_pickle=False)\n    offsets = np.load(\n        cache_dir / "well_offsets.int64.npy",\n        mmap_mode="r",\n        allow_pickle=False,\n    )\n    expected = int(metadata["rows"])\n    if features.shape != (expected, int(metadata["feature_count"])):\n        raise ValueError("feature cache shape does not match metadata")\n    if base.shape != (expected,) or truth.shape != (expected,):\n        raise ValueError("base/truth cache shape does not match metadata")\n    if offsets.shape != (int(metadata["wells"]) + 1,) or int(offsets[-1]) != expected:\n        raise ValueError("well offsets do not cover the cache")\n    return {\n        "metadata": metadata,\n        "features": features,\n        "base": base,\n        "truth": truth,\n        "offsets": offsets,\n    }\n\n\ndef _partition(metadata: dict[str, Any], smoke_wells: int) -> tuple[np.ndarray, np.ndarray]:\n    rows = metadata["per_well"]\n    validation = np.asarray(\n        [index for index, row in enumerate(rows) if row["folds"][PROTOCOL] == VALIDATION_FOLD],\n        dtype=np.int64,\n    )\n    validation_fingerprints = {rows[index]["typewell_fingerprint"] for index in validation}\n    training = np.asarray(\n        [\n            index\n            for index, row in enumerate(rows)\n            if row["folds"][PROTOCOL] != VALIDATION_FOLD\n            and row["typewell_fingerprint"] not in validation_fingerprints\n        ],\n        dtype=np.int64,\n    )\n    if smoke_wells:\n        training = training[:smoke_wells]\n        validation = validation[: max(1, smoke_wells // 4)]\n    if not len(training) or not len(validation):\n        raise ValueError("training/validation partition is empty")\n    return training, validation\n\n\ndef _normalization(\n    features: np.ndarray,\n    offsets: np.ndarray,\n    training_wells: np.ndarray,\n    *,\n    maximum_rows: int = 500_000,\n) -> tuple[np.ndarray, np.ndarray]:\n    lengths = offsets[training_wells + 1] - offsets[training_wells]\n    total = int(lengths.sum())\n    stride = max(1, math.ceil(total / maximum_rows))\n    samples: list[np.ndarray] = []\n    cursor = 0\n    for well_index, length in zip(training_wells, lengths, strict=True):\n        start = int(offsets[well_index])\n        local_offset = (-cursor) % stride\n        samples.append(\n            np.asarray(\n                features[start + local_offset : start + int(length) : stride],\n                dtype=np.float32,\n            )\n        )\n        cursor += int(length)\n    sample = np.concatenate(samples, axis=0)\n    center = np.nanmedian(sample, axis=0)\n    q25 = np.nanquantile(sample, 0.25, axis=0)\n    q75 = np.nanquantile(sample, 0.75, axis=0)\n    scale = (q75 - q25) / 1.349\n    center = np.where(np.isfinite(center), center, 0.0).astype(np.float32)\n    scale = np.where(np.isfinite(scale) & (scale > 1e-5), scale, 1.0).astype(np.float32)\n    return center, scale\n\n\ndef _window_batch(\n    rng: np.random.Generator,\n    features: np.ndarray,\n    base: np.ndarray,\n    truth: np.ndarray,\n    offsets: np.ndarray,\n    wells: np.ndarray,\n    center: np.ndarray,\n    scale: np.ndarray,\n    *,\n    batch_size: int,\n    window_rows: int,\n    well_sampling: str,\n) -> tuple[np.ndarray, np.ndarray, np.ndarray]:\n    lengths = offsets[wells + 1] - offsets[wells]\n    if well_sampling == "length":\n        probability = lengths.astype(np.float64)\n        probability /= probability.sum()\n    elif well_sampling == "uniform":\n        probability = None\n    else:\n        raise ValueError(f"unsupported well sampling: {well_sampling}")\n    selected = rng.choice(wells, size=batch_size, replace=True, p=probability)\n    batch_x = np.zeros((batch_size, len(center), window_rows), dtype=np.float32)\n    batch_y = np.zeros((batch_size, window_rows), dtype=np.float32)\n    batch_mask = np.zeros((batch_size, window_rows), dtype=np.float32)\n    for batch_index, well_index in enumerate(selected):\n        well_start = int(offsets[well_index])\n        well_stop = int(offsets[well_index + 1])\n        length = well_stop - well_start\n        local_start = int(rng.integers(0, max(1, length - window_rows + 1)))\n        stop = min(well_stop, well_start + local_start + window_rows)\n        rows = stop - (well_start + local_start)\n        values = np.asarray(\n            features[well_start + local_start : stop],\n            dtype=np.float32,\n        )\n        values = np.clip((values - center) / scale, -8.0, 8.0)\n        values[~np.isfinite(values)] = 0.0\n        residual = np.asarray(\n            truth[well_start + local_start : stop] - base[well_start + local_start : stop],\n            dtype=np.float32,\n        )\n        batch_x[batch_index, :, :rows] = values.T\n        batch_y[batch_index, :rows] = np.clip(residual, -TARGET_CLIP, TARGET_CLIP)\n        batch_mask[batch_index, :rows] = 1.0\n    return batch_x, batch_y, batch_mask\n\n\n@torch.inference_mode()\ndef _predict_well(\n    model: nn.Module,\n    features: np.ndarray,\n    center: np.ndarray,\n    scale: np.ndarray,\n    device: torch.device,\n) -> np.ndarray:\n    values = np.asarray(features, dtype=np.float32)\n    values = np.clip((values - center) / scale, -8.0, 8.0)\n    values[~np.isfinite(values)] = 0.0\n    tensor = torch.from_numpy(values.T[None]).to(device)\n    with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):\n        prediction = model(tensor)\n    return prediction.squeeze(0).float().cpu().numpy()\n\n\ndef _evaluate(\n    model: nn.Module,\n    cache: dict[str, Any],\n    validation_wells: np.ndarray,\n    center: np.ndarray,\n    scale: np.ndarray,\n    device: torch.device,\n) -> dict[str, Any]:\n    features = cache["features"]\n    base_values = cache["base"]\n    truth_values = cache["truth"]\n    offsets = cache["offsets"]\n    metadata = cache["metadata"]\n    per_well: list[dict[str, Any]] = []\n    all_truth: list[np.ndarray] = []\n    all_base: list[np.ndarray] = []\n    all_raw: list[np.ndarray] = []\n    model.eval()\n    for well_index in validation_wells:\n        start = int(offsets[well_index])\n        stop = int(offsets[well_index + 1])\n        truth = np.asarray(truth_values[start:stop], dtype=np.float32)\n        base = np.asarray(base_values[start:stop], dtype=np.float32)\n        raw = _predict_well(model, features[start:stop], center, scale, device)\n        all_truth.append(truth)\n        all_base.append(base)\n        all_raw.append(raw)\n        per_well.append(\n            {\n                "well": metadata["per_well"][well_index]["well"],\n                "rows": int(len(truth)),\n                "base_rmse": _rmse(truth, base),\n            }\n        )\n    truth = np.concatenate(all_truth)\n    base = np.concatenate(all_base)\n    raw = np.concatenate(all_raw)\n    scale_results: dict[str, Any] = {}\n    for candidate_scale in DIAGNOSTIC_SCALES:\n        prediction = base + candidate_scale * raw\n        well_cursor = 0\n        well_rmse: list[float] = []\n        for row in per_well:\n            stop = well_cursor + int(row["rows"])\n            well_rmse.append(_rmse(truth[well_cursor:stop], prediction[well_cursor:stop]))\n            well_cursor = stop\n        base_well_rmse = np.asarray([row["base_rmse"] for row in per_well], dtype=float)\n        candidate_well_rmse = np.asarray(well_rmse, dtype=float)\n        scale_results[f"{candidate_scale:.2f}"] = {\n            "pooled_rmse": _rmse(truth, prediction),\n            "pooled_delta": _rmse(truth, prediction) - _rmse(truth, base),\n            "macro_well_rmse": float(np.mean(candidate_well_rmse)),\n            "p95_well_delta": float(\n                np.quantile(candidate_well_rmse, 0.95) - np.quantile(base_well_rmse, 0.95)\n            ),\n            "worst_well_delta": float(\n                np.max(candidate_well_rmse) - np.max(base_well_rmse)\n            ),\n            "nonpositive_well_fraction": float(\n                np.mean(candidate_well_rmse <= base_well_rmse)\n            ),\n        }\n    primary = scale_results[f"{PRIMARY_SCALE:.2f}"]\n    checks = {\n        "pooled_delta_at_most_minus_0_02": bool(primary["pooled_delta"] <= -0.02),\n        "p95_not_worse_than_plus_0_10": bool(primary["p95_well_delta"] <= 0.10),\n        "worst_not_worse_than_plus_0_50": bool(primary["worst_well_delta"] <= 0.50),\n        "at_least_half_wells_nonpositive": bool(primary["nonpositive_well_fraction"] >= 0.50),\n    }\n    return {\n        "rows": int(len(truth)),\n        "wells": int(len(validation_wells)),\n        "base_pooled_rmse": _rmse(truth, base),\n        "raw_residual": {\n            "mean": float(np.mean(raw)),\n            "std": float(np.std(raw)),\n            "p95_abs": float(np.quantile(np.abs(raw), 0.95)),\n            "maximum_abs": float(np.max(np.abs(raw))),\n        },\n        "scale_results": scale_results,\n        "primary_scale": PRIMARY_SCALE,\n        "gate": {\n            "decision": "PASS" if all(checks.values()) else "FAIL",\n            "checks": checks,\n        },\n    }\n\n\ndef main() -> int:\n    args = _parser().parse_args()\n    if args.steps <= 0 or args.batch_size <= 0 or args.window_rows < 32:\n        raise ValueError("training controls are invalid")\n    started = time.perf_counter()\n    random.seed(RANDOM_SEED)\n    np.random.seed(RANDOM_SEED)\n    torch.manual_seed(RANDOM_SEED)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(RANDOM_SEED)\n    if args.device == "auto":\n        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    else:\n        device = torch.device(args.device)\n    if device.type == "cuda" and not torch.cuda.is_available():\n        raise RuntimeError("CUDA was requested but is unavailable")\n    print(f"device={device} torch={torch.__version__}", flush=True)\n\n    cache = _load_cache(args.cache_dir.expanduser().resolve())\n    training_wells, validation_wells = _partition(cache["metadata"], args.smoke_wells)\n    center, feature_scale = _normalization(\n        cache["features"],\n        cache["offsets"],\n        training_wells,\n        maximum_rows=50_000 if args.smoke_wells else 500_000,\n    )\n    model = SequenceResidualTCN(len(center)).to(device)\n    optimizer = torch.optim.AdamW(\n        model.parameters(),\n        lr=LEARNING_RATE,\n        weight_decay=WEIGHT_DECAY,\n    )\n    loss_function = nn.SmoothL1Loss(reduction="none", beta=2.0)\n    rng = np.random.default_rng(RANDOM_SEED)\n    losses: list[float] = []\n    model.train()\n    for step in range(args.steps):\n        batch_x, batch_y, batch_mask = _window_batch(\n            rng,\n            cache["features"],\n            cache["base"],\n            cache["truth"],\n            cache["offsets"],\n            training_wells,\n            center,\n            feature_scale,\n            batch_size=args.batch_size,\n            window_rows=args.window_rows,\n            well_sampling=args.well_sampling,\n        )\n        values = torch.from_numpy(batch_x).to(device)\n        target = torch.from_numpy(batch_y).to(device)\n        mask = torch.from_numpy(batch_mask).to(device)\n        optimizer.zero_grad(set_to_none=True)\n        with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):\n            prediction = model(values)\n            loss = (loss_function(prediction, target) * mask).sum() / mask.sum()\n        loss.backward()\n        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)\n        optimizer.step()\n        losses.append(float(loss.detach().cpu()))\n        if (step + 1) % max(1, min(250, args.steps // 4)) == 0:\n            print(\n                f"[train {step + 1:4d}/{args.steps}] "\n                f"loss={np.mean(losses[-100:]):.5f} "\n                f"elapsed={time.perf_counter() - started:.1f}s",\n                flush=True,\n            )\n\n    evaluation = _evaluate(\n        model,\n        cache,\n        validation_wells,\n        center,\n        feature_scale,\n        device,\n    )\n    report = {\n        "contract": {\n            "writes_submission": False,\n            "protocol": PROTOCOL,\n            "validation_fold": VALIDATION_FOLD,\n            "validation_typewell_fingerprints_excluded_from_training": True,\n            "well_id_encoded_as_feature": False,\n            "absolute_position_features_removed": True,\n            "training_well_sampling": args.well_sampling,\n            "primary_scale_frozen_before_training": PRIMARY_SCALE,\n            "diagnostic_scales": list(DIAGNOSTIC_SCALES),\n            "primary_gate": {\n                "pooled_delta_limit": -0.02,\n                "p95_well_delta_limit": 0.10,\n                "worst_well_delta_limit": 0.50,\n                "minimum_nonpositive_well_fraction": 0.50,\n            },\n        },\n        "runtime": {\n            "device": str(device),\n            "torch": torch.__version__,\n            "cuda_device": (\n                torch.cuda.get_device_name(0) if device.type == "cuda" else None\n            ),\n            "seconds": float(time.perf_counter() - started),\n        },\n        "training": {\n            "wells": int(len(training_wells)),\n            "steps": int(args.steps),\n            "batch_size": int(args.batch_size),\n            "window_rows": int(args.window_rows),\n            "well_sampling": args.well_sampling,\n            "learning_rate": LEARNING_RATE,\n            "weight_decay": WEIGHT_DECAY,\n            "target_clip": TARGET_CLIP,\n            "output_bound": OUTPUT_BOUND,\n            "final_loss_mean_100": float(np.mean(losses[-100:])),\n            "smoke_wells": int(args.smoke_wells),\n        },\n        "validation": evaluation,\n        "decision": {\n            "decision": evaluation["gate"]["decision"],\n            "full_multifold_authorized": evaluation["gate"]["decision"] == "PASS"\n            and not args.smoke_wells,\n            "kaggle_submission_authorized": False,\n        },\n    }\n    output = args.output.expanduser().resolve()\n    output.parent.mkdir(parents=True, exist_ok=True)\n    output.write_text(\n        json.dumps(_json_safe(report), indent=2, sort_keys=True) + "\\n",\n        encoding="utf-8",\n    )\n    checkpoint = args.checkpoint.expanduser().resolve()\n    checkpoint.parent.mkdir(parents=True, exist_ok=True)\n    torch.save(\n        {\n            "state_dict": model.state_dict(),\n            "center": center,\n            "scale": feature_scale,\n            "feature_names": cache["metadata"]["features"],\n            "contract": report["contract"],\n        },\n        checkpoint,\n    )\n    primary = evaluation["scale_results"][f"{PRIMARY_SCALE:.2f}"]\n    print(\n        f"decision={evaluation[\'gate\'][\'decision\']} "\n        f"base={evaluation[\'base_pooled_rmse\']:.6f} "\n        f"primary={primary[\'pooled_rmse\']:.6f} "\n        f"delta={primary[\'pooled_delta\']:+.6f}",\n        flush=True,\n    )\n    print(f"wrote {output} and {checkpoint}", flush=True)\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'
script = Path('/kaggle/working/sequence_tcn_pilot.py')
script.write_text(source, encoding='utf-8')
command = [
    sys.executable, str(script),
    '--cache-dir', str(cache),
    '--output', '/kaggle/working/sequence_tcn_balanced_pilot_report.json',
    '--checkpoint', '/kaggle/working/sequence_tcn_balanced_pilot.pt',
    '--well-sampling', 'uniform',
    '--device', 'cuda',
]
print('running:', ' '.join(command), flush=True)
subprocess.run(command, check=True)
report_path = Path(command[command.index('--output') + 1])
report = json.loads(report_path.read_text())
if report['training']['well_sampling'] != 'uniform':
    raise RuntimeError('balanced pilot did not use uniform sampling')
print(json.dumps({
    'decision': report['decision'],
    'runtime': report['runtime'],
    'training': report['training'],
    'validation': report['validation'],
}, indent=2, sort_keys=True))
